In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split


In [ ]:
np.random.seed(42)
tf.random.set_seed(42)

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 30

In [ ]:
base_dir = './dataset/'
class_names = sorted(
    d for d in os.listdir(base_dir)
    if os.path.isdir(os.path.join(base_dir, d))
)
num_classes = len(class_names)
label_to_index = {name: i for i, name in enumerate(class_names)}

image_paths = []
labels = []

In [ ]:
for class_name in class_names:
    class_dir = os.path.join(base_dir, class_name)
    for image_name in os.listdir(class_dir):
        image_path = os.path.join(class_dir, image_name)
        image_paths.append(image_path)
        labels.append(class_name)

df = pd.DataFrame({'image_path': image_paths, 'label': labels})

train_df, test_df = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=42)
train_df, val_df = train_test_split(train_df, test_size=0.2, stratify=train_df['label'], random_state=42)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

In [ ]:
def build_dataset(df, shuffle=False, augment=False):
    paths = df['image_path'].astype(str).values
    y = np.array([label_to_index[l] for l in df['label']], dtype=np.int32)
    ds = tf.data.Dataset.from_tensor_slices((paths, y))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(df), seed=42, reshuffle_each_iteration=True)

    def decode(path, label):
        img = tf.io.read_file(path)
        img = tf.io.decode_image(img, channels=3, expand_animations=False)
        img.set_shape([None, None, 3])
        img = tf.image.resize(img, IMG_SIZE)
        img = tf.cast(img, tf.float32) / 255.0
        return img, label

    ds = ds.map(decode, num_parallel_calls=tf.data.AUTOTUNE)

    if augment:
        def aug(image, label):
            image = tf.image.random_flip_left_right(image)
            image = tf.image.random_brightness(image, max_delta=0.2)
            image = tf.image.random_saturation(image, 0.8, 1.2)
            return tf.clip_by_value(image, 0.0, 1.0), label

        ds = ds.map(aug, num_parallel_calls=tf.data.AUTOTUNE)

    return ds


train_ds = build_dataset(train_df, shuffle=True, augment=True).batch(BATCH_SIZE).repeat().prefetch(
    tf.data.AUTOTUNE
)
val_ds = build_dataset(val_df, shuffle=False, augment=False).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_ds = build_dataset(test_df, shuffle=False, augment=False).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [ ]:
model = models.Sequential([
    layers.Input(shape=(*IMG_SIZE, 3)),
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(num_classes, activation='softmax'),
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

In [ ]:
history = model.fit(
    train_ds,
    steps_per_epoch=max(1, len(train_df) // BATCH_SIZE),
    epochs=EPOCHS,
    validation_data=val_ds,
    validation_steps=max(1, len(val_df) // BATCH_SIZE),
)

In [ ]:
test_loss, test_accuracy = model.evaluate(test_ds)
print(f"Test accuracy: {test_accuracy:.4f}")